# aDDM Tutorial

This notebook showcases the implementation of a modern aDDM, compatible with PyDDM.

### Load the data

In [ ]:
from ast import literal_eval
import pandas as pd

# 1. Load data
df_raw = pd.read_csv('1ms_trial_data.csv')

# 2. Drop nuisance trials
to_drop = pd.read_csv("dropped_trials.csv").rename(columns={"parcode": "sub_id"})

df = df_raw.loc[
    ~df_raw.set_index(["sub_id", "trial"]).index.isin(
        to_drop.set_index(["sub_id", "trial"]).index
    )
    & (~df_raw["hidden"])
]

# 3. Adjustments
df['RT'] = (df['RT']*1000).astype(int) # RT unit scaling
df['fixation'] = df['fixation'].apply(literal_eval) # String to list as a result of csv saving
df['choice'] = df['choice'].replace({"left": 0, "right": 1}) # Map choice to 0 or 1

### Simulating data from empiricals

In [ ]:
from simulation import get_corrected_empirical_distributions
import numpy as np

# Make empirical distributions
# value_diffs = np.arange(-4, 4.25, 0.25)
value_diffs = np.unique(df['avgWTP_left'] - df['avgWTP_right'])
legend = {
    "left": {1},
    "right": {2},
    "transition": {0}, 
    "blank_fixation": {4}
}
fixation_col = 'fixation'
left_value_col = 'avgWTP_left'
right_value_col = 'avgWTP_right'

empirical_distributions = get_corrected_empirical_distributions(
    df,
    value_diffs=value_diffs,
    legend=legend,
    fixation_col=fixation_col,
    left_value_col=left_value_col,
    right_value_col=right_value_col,
    cutoff=0.9
)

In [ ]:
from simulation import generate_fixations

# Create sample trial conditions
dt = 0.01
seed = 42

trials = df.loc[
    (df['sub_id'] == 304) & (df['trial'] % 2 == 1),
    ['avgWTP_left', 'avgWTP_right']
].copy()
trials['fixation'] = None

rng = np.random.default_rng(seed)
trials_dict = []
for idx, r in trials.iterrows():
    fx = generate_fixations(
        dt, 
        r.avgWTP_left - r.avgWTP_right, 
        empirical_distributions,
        rng=rng
    )
    if fx is not None:
        trials_dict.append({
            "avgWTP_left": r.avgWTP_left,
            "avgWTP_right": r.avgWTP_right,
            "fixation": fx
        })

In [ ]:
from simulation import simulate
import pyddm

model_conditions = {'drift_rate': 0.8, 'theta': 0.5, 'noise': 0.6}

results_df = simulate(dt, model_conditions, trials_dict, seed=seed, save_results=False)
# results_df['sub_id'] = f'seed{seed}_subjects{size}_sim'
# results_df['trial'] = range(1, len(trials_clean) + 1)
# results_df = results_df.rename(columns={'fixation': 'fix_sequence'})
results_df = results_df.drop(columns = ['trajectory'])

sample = pyddm.Sample.from_pandas_dataframe(
    results_df,
    choice_column_name="choice",
    rt_column_name="RT",
    choice_names=("left", "right")
)

print(f'Average RT: {results_df["RT"].mean():.2f} seconds (out of {len(results_df)} trials)')
results_df.head()

The above is the first half of the tutorial. Following is native parameter recovery by differential evolution.

In [ ]:
# Define the model
def drift_function(avgWTP_left, avgWTP_right, fixation, d, x, t):
        fixation_index = min(int(t/dt), len(fixation)-1)
        current_fixation = fixation[fixation_index]
        if current_fixation == 0: # saccade
            drift_val = 0
        elif current_fixation == 1: # left
            drift_val = d * (avgWTP_left - avgWTP_right * model_conditions['theta'])
        else: # right
            drift_val = d * (avgWTP_left * model_conditions['theta'] - avgWTP_right)
        
        return np.ones_like(x) * drift_val
    
def noise_function(n, x, t):
    return np.ones_like(x) * n

model = pyddm.gddm(
    drift=drift_function,
    noise=noise_function,
    bound=1,
    nondecision=0,
    parameters={'d': (0.7, 0.9), 'n': (0.5, 0.7)},
    conditions=["avgWTP_left", "avgWTP_right", "fixation"],
    choice_names=("left", "right"),
    T_dur=30,
    dx=0.01,
    dt=dt
)

model._overlay = pyddm.models.OverlayChain(overlays=[])

model.fit(sample=sample, verbose=True)